In [1]:
# ══════════════════════════════════════════════════════════
# CELL 1: INSTALL & DOWNLOAD BOTH DATASETS
# ══════════════════════════════════════════════════════════
!pip install roboflow ultralytics -q

from roboflow import Roboflow

API_KEY = "myVnKSojtknTKBT9Rn1z"   # ← your existing key
rf      = Roboflow(api_key=API_KEY)

# ── Dataset 1: Obstacle Detection  ──────────────
print("Downloading Dataset 1...")
dataset1 = rf.workspace("-cmbyf") \
              .project("obstacle-detection-yeuzf-jffqd") \
              .version(1) \
              .download("yolov8",
                        location="/kaggle/working/yolo_blind_assist/dataset1")

# ── Dataset 2: NEW — detai_dataset1 ───────────────────────
print("\nDownloading Dataset 2 (new)...")
dataset2 = rf.workspace("ahmets-workspace-hgqwt") \
              .project("detai_dataset1") \
              .version(1) \
              .download("yolov8",
                        location="/kaggle/working/yolo_blind_assist/dataset2")

print("\n✓ Both datasets downloaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/yolo_blind_assist/dataset1 in yolov8:: 100%|██████████| 18371/18371 [00:02<00:00, 6535.83it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/yolo_blind_assist/dataset2 in yolov8:: 100%|██████████| 81239/81239 [00:11<00:00, 7225.57it/s] 



✓ Both datasets downloaded


In [2]:
# ══════════════════════════════════════════════════════════
# CELL 2: INSPECT BOTH DATASETS
# ══════════════════════════════════════════════════════════
import yaml
from pathlib import Path

DATASET1_DIR = Path('/kaggle/working/yolo_blind_assist/dataset1')
DATASET2_DIR = Path('/kaggle/working/yolo_blind_assist/dataset2')

def inspect_dataset(dataset_dir, name):
    print(f"\n{'='*55}")
    print(f"DATASET: {name}")
    print(f"{'='*55}")

    yaml_files = list(dataset_dir.rglob('*.yaml'))
    if not yaml_files:
        print("✗ No YAML found!")
        return None

    with open(yaml_files[0]) as f:
        cfg = yaml.safe_load(f)

    names = cfg.get('names', [])
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names.keys())]

    print(f"Classes ({cfg.get('nc', '?')}):")
    for i, n in enumerate(names):
        print(f"  {i:3d}: {n}")

    print(f"\nSplits:")
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = dataset_dir / split / 'images'
        if img_dir.exists():
            print(f"  {split:8s}: {len(list(img_dir.glob('*')))} images")

    return cfg

cfg1 = inspect_dataset(DATASET1_DIR, "Dataset 1 - Obstacle Detection")
cfg2 = inspect_dataset(DATASET2_DIR, "Dataset 2 - detai_dataset1")


DATASET: Dataset 1 - Obstacle Detection
Classes (10):
    0: Bicycle
    1: Bus
    2: Car
    3: Dog
    4: Electric pole
    5: Motorcycle
    6: Person
    7: Traffic signs
    8: Tree
    9: Uncovered manhole

Splits:
  train   : 7855 images
  valid   : 865 images
  test    : 463 images

DATASET: Dataset 2 - detai_dataset1
Classes (17):
    0: bench
    1: bicycle
    2: bus
    3: car
    4: cat
    5: chair
    6: crosswalk
    7: dog
    8: door
    9: motorcycle
   10: person
   11: pole
   12: sidewalk
   13: stair
   14: stop sign
   15: traffic light
   16: truck

Splits:
  train   : 28431 images
  valid   : 8125 images
  test    : 4061 images


In [3]:
# ══════════════════════════════════════════════════════════
# CELL 3: DEFINE CLASS MAPPINGS — FULLY DYNAMIC
# Reads from downloaded dataset yamls automatically.
# No hardcoding needed — handles any new dataset.
# ══════════════════════════════════════════════════════════
import yaml
from pathlib import Path

# ── COCO 80 classes (indices 0–79) ────────────────────────
COCO_CLASSES = [
    'person','bicycle','car','motorcycle','airplane','bus',
    'train','truck','boat','traffic light','fire hydrant',
    'stop sign','parking meter','bench','bird','cat','dog',
    'horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee',
    'skis','snowboard','sports ball','kite','baseball bat',
    'baseball glove','skateboard','surfboard','tennis racket',
    'bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot',
    'hot dog','pizza','donut','cake','chair','couch',
    'potted plant','bed','dining table','toilet','tv','laptop',
    'mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase',
    'scissors','teddy bear','hair drier','toothbrush'
]

# Lookup: lowercase name → COCO id (for case-insensitive matching)
COCO_LOWER = {name.lower(): i for i, name in enumerate(COCO_CLASSES)}

# ── Auto-detect new classes from both dataset yamls ────────
def get_class_names(dataset_dir):
    yaml_files = list(Path(dataset_dir).rglob('*.yaml'))
    if not yaml_files:
        raise FileNotFoundError(f"No yaml in {dataset_dir}")
    with open(yaml_files[0]) as f:
        cfg = yaml.safe_load(f)
    names = cfg.get('names', [])
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names.keys())]
    return names

classes1 = get_class_names(DATASET1_DIR)
classes2 = get_class_names(DATASET2_DIR)

# Collect truly new classes across both datasets
# (anything not in COCO, preserving order, no duplicates)
TRULY_NEW_CLASSES = []
for cls in classes1 + classes2:
    if cls.lower() not in COCO_LOWER \
            and cls not in TRULY_NEW_CLASSES:
        TRULY_NEW_CLASSES.append(cls)

ALL_CLASSES = COCO_CLASSES + TRULY_NEW_CLASSES
NC_TOTAL    = len(ALL_CLASSES)

# New-class lookup: lowercase → merged id (80+)
NEW_LOWER = {cls.lower(): 80 + i
             for i, cls in enumerate(TRULY_NEW_CLASSES)}

print("=" * 55)
print("CLASS MAPPING SUMMARY")
print("=" * 55)
print(f"COCO classes  : 80  (IDs 0–79)")
print(f"New classes   : {len(TRULY_NEW_CLASSES)}  "
      f"(IDs 80–{NC_TOTAL-1})")
print(f"Total classes : {NC_TOTAL}")
print()
print("New classes auto-detected:")
for i, cls in enumerate(TRULY_NEW_CLASSES):
    print(f"  [{80+i:3d}] {cls}")

CLASS MAPPING SUMMARY
COCO classes  : 80  (IDs 0–79)
New classes   : 9  (IDs 80–88)
Total classes : 89

New classes auto-detected:
  [ 80] Electric pole
  [ 81] Traffic signs
  [ 82] Tree
  [ 83] Uncovered manhole
  [ 84] crosswalk
  [ 85] door
  [ 86] pole
  [ 87] sidewalk
  [ 88] stair


In [4]:
# ══════════════════════════════════════════════════════════
# CELL 4: BUILD REMAP DICTS FOR BOTH DATASETS
# ══════════════════════════════════════════════════════════

def build_remap(dataset_dir, dataset_name):
    """
    Maps each dataset class id → final merged class id.
    COCO overlap  → 0–79
    New classes   → 80+
    Unknown       → skipped (won't appear in training)
    """
    names = get_class_names(dataset_dir)

    print(f"\n{dataset_name} remap:")
    print("-" * 50)

    remap   = {}
    skipped = []

    for old_id, cls in enumerate(names):
        cls_lower = cls.lower()

        if cls_lower in COCO_LOWER:
            new_id = COCO_LOWER[cls_lower]
            remap[old_id] = new_id
            print(f"  [{old_id:2d}] '{cls}' "
                  f"→ COCO [{new_id:2d}] '{COCO_CLASSES[new_id]}'")

        elif cls_lower in NEW_LOWER:
            new_id = NEW_LOWER[cls_lower]
            remap[old_id] = new_id
            print(f"  [{old_id:2d}] '{cls}' "
                  f"→ NEW  [{new_id:2d}]")

        else:
            skipped.append((old_id, cls))
            print(f"  [{old_id:2d}] '{cls}' → ⚠ SKIPPED")

    if skipped:
        print(f"\n  ⚠ {len(skipped)} class(es) skipped.")
        print("  → Add them to TRULY_NEW_CLASSES in Cell 3 "
              "if you want them included.")

    return remap

REMAP_DS1 = build_remap(DATASET1_DIR, "Dataset 1")
REMAP_DS2 = build_remap(DATASET2_DIR, "Dataset 2")

print(f"\n{'='*55}")
print(f"DS1 remap: {REMAP_DS1}")
print(f"DS2 remap: {REMAP_DS2}")


Dataset 1 remap:
--------------------------------------------------
  [ 0] 'Bicycle' → COCO [ 1] 'bicycle'
  [ 1] 'Bus' → COCO [ 5] 'bus'
  [ 2] 'Car' → COCO [ 2] 'car'
  [ 3] 'Dog' → COCO [16] 'dog'
  [ 4] 'Electric pole' → NEW  [80]
  [ 5] 'Motorcycle' → COCO [ 3] 'motorcycle'
  [ 6] 'Person' → COCO [ 0] 'person'
  [ 7] 'Traffic signs' → NEW  [81]
  [ 8] 'Tree' → NEW  [82]
  [ 9] 'Uncovered manhole' → NEW  [83]

Dataset 2 remap:
--------------------------------------------------
  [ 0] 'bench' → COCO [13] 'bench'
  [ 1] 'bicycle' → COCO [ 1] 'bicycle'
  [ 2] 'bus' → COCO [ 5] 'bus'
  [ 3] 'car' → COCO [ 2] 'car'
  [ 4] 'cat' → COCO [15] 'cat'
  [ 5] 'chair' → COCO [56] 'chair'
  [ 6] 'crosswalk' → NEW  [84]
  [ 7] 'dog' → COCO [16] 'dog'
  [ 8] 'door' → NEW  [85]
  [ 9] 'motorcycle' → COCO [ 3] 'motorcycle'
  [10] 'person' → COCO [ 0] 'person'
  [11] 'pole' → NEW  [86]
  [12] 'sidewalk' → NEW  [87]
  [13] 'stair' → NEW  [88]
  [14] 'stop sign' → COCO [11] 'stop sign'
  [15] 'traffic

In [5]:
# ══════════════════════════════════════════════════════════
# CELL 5: COPY & REMAP BOTH DATASETS INTO MERGED FOLDER
# ══════════════════════════════════════════════════════════
import shutil
from pathlib import Path

MERGED_DIR = Path('/kaggle/working/merged_dataset')

for split in ['train', 'val']:
    (MERGED_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print(f"✓ Merged dataset folder ready: {MERGED_DIR}")

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def remap_label_file(src_path, dst_path, remap):
    """Remap class IDs in a YOLO label file. Returns True if any lines written."""
    lines_out = []
    with open(src_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            old_id = int(parts[0])
            if old_id not in remap:
                continue
            parts[0] = str(remap[old_id])
            lines_out.append(' '.join(parts))
    if lines_out:
        with open(dst_path, 'w') as f:
            f.write('\n'.join(lines_out))
        return True
    return False


def copy_dataset(dataset_dir, merged_dir, remap,
                 prefix, name):
    dataset_dir = Path(dataset_dir)
    merged_dir  = Path(merged_dir)
    total_imgs = total_lbls = skipped = 0

    print(f"\nCopying: {name}")
    print("-" * 50)

    # 'valid' is Roboflow's name for val split
    for src_split, dst_split in [('train','train'),
                                  ('valid','val'),
                                  ('val',  'val')]:
        img_src = dataset_dir / src_split / 'images'
        lbl_src = dataset_dir / src_split / 'labels'  # ← fixed: always separate

        if not img_src.exists():
            continue

        img_dst = merged_dir / 'images' / dst_split
        lbl_dst = merged_dir / 'labels' / dst_split

        imgs_n = lbls_n = 0

        for img_path in img_src.iterdir():
            if img_path.suffix.lower() not in IMG_EXTS:
                continue

            dst_img = img_dst / f"{prefix}_{img_path.name}"
            shutil.copy2(img_path, dst_img)
            imgs_n += 1

            lbl_path     = lbl_src / (img_path.stem + '.txt')
            dst_lbl_path = lbl_dst / f"{prefix}_{img_path.stem}.txt"

            if lbl_path.exists():
                ok = remap_label_file(lbl_path, dst_lbl_path, remap)
                if ok:
                    lbls_n += 1
                else:
                    skipped += 1   # label existed but all classes unmapped

        print(f"  {src_split:6s} → {dst_split:5s}: "
              f"{imgs_n:5d} images | {lbls_n:5d} labels")
        total_imgs += imgs_n
        total_lbls += lbls_n

    if skipped:
        print(f"  ⚠ {skipped} labels skipped (all classes unmapped)")
    print(f"  Subtotal: {total_imgs} images | {total_lbls} labels ✓")
    return total_imgs, total_lbls


imgs1, lbls1 = copy_dataset(DATASET1_DIR, MERGED_DIR,
                             REMAP_DS1, 'ds1',
                             'Dataset 1 – Obstacle Detection')
imgs2, lbls2 = copy_dataset(DATASET2_DIR, MERGED_DIR,
                             REMAP_DS2, 'ds2',
                             'Dataset 2 – detai_dataset1')

print(f"\n{'='*55}")
print(f"BOTH DATASETS MERGED")
print(f"{'='*55}")
print(f"  DS1 : {imgs1:5d} images | {lbls1:5d} labels")
print(f"  DS2 : {imgs2:5d} images | {lbls2:5d} labels")
print(f"  TOTAL: {imgs1+imgs2:4d} images | {lbls1+lbls2:4d} labels")

✓ Merged dataset folder ready: /kaggle/working/merged_dataset

Copying: Dataset 1 – Obstacle Detection
--------------------------------------------------
  train  → train:  7855 images |  7737 labels
  valid  → val  :   865 images |   849 labels
  ⚠ 134 labels skipped (all classes unmapped)
  Subtotal: 8720 images | 8586 labels ✓

Copying: Dataset 2 – detai_dataset1
--------------------------------------------------
  train  → train: 28431 images | 26949 labels
  valid  → val  :  8125 images |  7331 labels
  ⚠ 2276 labels skipped (all classes unmapped)
  Subtotal: 36556 images | 34280 labels ✓

BOTH DATASETS MERGED
  DS1 :  8720 images |  8586 labels
  DS2 : 36556 images | 34280 labels
  TOTAL: 45276 images | 42866 labels


In [6]:
# ══════════════════════════════════════════════════════════
# CELL 6: ADD COCO SUBSET 
# ══════════════════════════════════════════════════════════
import os, random, shutil, time
import urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import subprocess

WORK_DIR          = Path('/kaggle/working')
MERGED_DIR        = WORK_DIR / 'merged_dataset'
COCO_BASE         = WORK_DIR / 'coco'

COCO_TRAIN_LBL    = COCO_BASE / 'labels' / 'train2017'
COCO_VAL_LBL      = COCO_BASE / 'labels' / 'val2017'
COCO_VAL_IMG_DIR  = COCO_BASE / 'images'  / 'val2017'

TRAIN_IMG_DST     = MERGED_DIR / 'images' / 'train'
TRAIN_LBL_DST     = MERGED_DIR / 'labels' / 'train'
VAL_IMG_DST       = MERGED_DIR / 'images' / 'val'
VAL_LBL_DST       = MERGED_DIR / 'labels' / 'val'

COCO_TRAIN_SAMPLE = 60_000
MAX_WORKERS       = 24

for p in [TRAIN_IMG_DST, TRAIN_LBL_DST, VAL_IMG_DST, VAL_LBL_DST]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print(" COCO SUBSET DOWNLOAD & MERGE")
print(f" Target: {COCO_TRAIN_SAMPLE:,} train + 5,000 val")
print("=" * 60)

# ── Clean up any bad nested folders from previous runs ────
for bad in [COCO_BASE / 'coco', WORK_DIR / 'coco' / 'coco']:
    if bad.exists():
        shutil.rmtree(bad, ignore_errors=True)
        print(f"Cleaned up nested folder: {bad}")

# ── Step 1: COCO YOLO labels (~46 MB) ─────────────────────
LABELS_URL = ('https://github.com/ultralytics/yolov5/releases/'
              'download/v1.0/coco2017labels.zip')
LABELS_ZIP = WORK_DIR / 'coco2017labels.zip'

existing = list(COCO_TRAIN_LBL.glob('*.txt')) \
           if COCO_TRAIN_LBL.exists() else []

if len(existing) < 100:
    print("\n[1/3] Downloading COCO YOLO labels (~46 MB)...")
    COCO_BASE.mkdir(parents=True, exist_ok=True)
    subprocess.run(['wget', '-q', '--show-progress',
                    LABELS_URL, '-O', str(LABELS_ZIP)], check=True)
    subprocess.run(['unzip', '-q', str(LABELS_ZIP),
                    '-d', str(WORK_DIR)], check=True)
    LABELS_ZIP.unlink(missing_ok=True)
    # ← fixed: compute count before f-string (no backslash in f-string)
    n_train_labels = len(list(COCO_TRAIN_LBL.glob('*.txt')))
    print(f"✓ Labels ready ({n_train_labels:,} train labels)")
else:
    print(f"[1/3] ✓ Labels exist ({len(existing):,} found) — skipped")

# ── Step 2: Download & merge val2017 images (~1 GB) ───────
VAL_ZIP = WORK_DIR / 'val2017.zip'

# Check folder exists AND actually contains images
val_jpgs = list(COCO_VAL_IMG_DIR.glob('*.jpg')) \
           if COCO_VAL_IMG_DIR.exists() else []

if not val_jpgs:
    print("\n[2/3] Downloading COCO val2017 images (~1 GB)...")
    subprocess.run(['wget', '-q', '--show-progress',
                    'http://images.cocodataset.org/zips/val2017.zip',
                    '-O', str(VAL_ZIP)], check=True)
    (COCO_BASE / 'images').mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', str(VAL_ZIP),
                    '-d', str(COCO_BASE / 'images')], check=True)
    VAL_ZIP.unlink(missing_ok=True)
    val_jpgs = list(COCO_VAL_IMG_DIR.glob('*.jpg'))
    print(f"✓ val2017 downloaded ({len(val_jpgs):,} images)")
else:
    print(f"[2/3] ✓ val2017 exists ({len(val_jpgs):,} images) — skipped")

# Merge val2017 into merged dataset val split
print("      Merging val2017 into val split...")
val_copied = 0
for img_path in tqdm(val_jpgs, desc="VAL merge"):
    lbl_path = COCO_VAL_LBL / f"{img_path.stem}.txt"
    if not lbl_path.exists():
        continue
    dst_img = VAL_IMG_DST / f"coco_{img_path.name}"
    dst_lbl = VAL_LBL_DST / f"coco_{lbl_path.name}"
    if not dst_img.exists():
        shutil.copy2(img_path, dst_img)
    if not dst_lbl.exists():
        shutil.copy2(lbl_path, dst_lbl)
    val_copied += 1
print(f"✓ val2017 merged ({val_copied:,} images)")

# ── Step 3: Targeted download of 30k train2017 images ─────
all_train_labels = list(COCO_TRAIN_LBL.glob('*.txt'))
if not all_train_labels:
    raise RuntimeError(
        "Training labels not found. "
        "Check the labels download completed correctly."
    )

random.seed(42)
selected = random.sample(
    all_train_labels,
    min(COCO_TRAIN_SAMPLE, len(all_train_labels))
)

# Resume-safe: skip already-downloaded images
to_download = [
    lbl for lbl in selected
    if not (TRAIN_IMG_DST / f"coco_{lbl.stem}.jpg").exists()
]
already_done = len(selected) - len(to_download)

if already_done:
    print(f"\n[3/3] {already_done:,} already downloaded — "
          f"fetching remaining {len(to_download):,}...")
else:
    print(f"\n[3/3] Downloading {len(to_download):,} "
          f"train2017 images ...")

def download_one(lbl_path, retries=3):
    """Download one COCO train image with retry + back-off."""
    img_name = f"{lbl_path.stem}.jpg"
    url      = f"http://images.cocodataset.org/train2017/{img_name}"
    dst_img  = TRAIN_IMG_DST / f"coco_{img_name}"
    dst_lbl  = TRAIN_LBL_DST / f"coco_{lbl_path.name}"

    if dst_img.exists():
        return True

    for attempt in range(retries):
        try:
            urllib.request.urlretrieve(url, dst_img)
            if not dst_lbl.exists():
                shutil.copy2(lbl_path, dst_lbl)
            return True
        except Exception:
            if attempt < retries - 1:
                time.sleep(0.5 * (attempt + 1))
            elif dst_img.exists():
                dst_img.unlink()  # remove partial file
    return False

success = failed = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(download_one, lbl): lbl
               for lbl in to_download}
    for future in tqdm(as_completed(futures),
                       total=len(futures),
                       desc="TRAIN download"):
        if future.result():
            success += 1
        else:
            failed += 1

print(f"✓ Downloaded: {success + already_done:,} | Failed: {failed:,}")
if failed > 200:
    print("⚠ High failure count — re-run cell (resume-safe)")

# ── Final summary ──────────────────────────────────────────
print("\n" + "=" * 60)
print(" FINAL DATASET SUMMARY")
print("=" * 60)
total_train = 0
for split in ['train', 'val']:
    n_imgs  = len(list((MERGED_DIR / 'images' / split).glob('*')))
    n_lbls  = len(list((MERGED_DIR / 'labels' / split).glob('*.txt')))
    n_coco  = len(list((MERGED_DIR / 'images' / split).glob('coco_*')))
    n_new   = n_imgs - n_coco
    if split == 'train':
        total_train = n_imgs
    balance = "✓" if abs(n_imgs - n_lbls) <= 10 else "⚠ MISMATCH"
    print(f"\n  {split.upper()} {balance}")
    print(f"    Total  : {n_imgs:>7,}  |  labels: {n_lbls:>7,}")
    print(f"    ├─ COCO: {n_coco:>7,}")
    print(f"    └─ New : {n_new:>7,}")

mins_per_epoch = (total_train / 32) * 0.10 / 60
print(f"\n  ⏱  T4 x2, batch=32, imgsz=640, YOLOv8n")
print(f"     ~{mins_per_epoch:.0f} min/epoch  |  "
      f"100 epochs ≈ {mins_per_epoch * 100 / 60:.1f} hrs")
status = ("✅ Fits in one session"
          if mins_per_epoch * 100 / 60 < 11
          else "⚠ May need resume=True")
print(f"     {status}")
print("=" * 60)

 COCO SUBSET DOWNLOAD & MERGE
 Target: 60,000 train + 5,000 val

[1/3] Downloading COCO YOLO labels (~46 MB)...



     0K .......... .......... .......... .......... ..........  0% 4.47M 10s
    50K .......... .......... .......... .......... ..........  0% 10.7M 7s
   100K .......... .......... .......... .......... ..........  0% 7.32M 7s
   150K .......... .......... .......... .......... ..........  0% 22.8M 6s
   200K .......... .......... .......... .......... ..........  0% 32.7M 5s
   250K .......... .......... .......... .......... ..........  0% 7.33M 5s
   300K .......... .......... .......... .......... ..........  0% 56.4M 4s
   350K .......... .......... .......... .......... ..........  0% 27.8M 4s
   400K .......... .......... .......... .......... ..........  0% 46.9M 4s
   450K .......... .......... .......... .......... ..........  1% 83.7M 3s
   500K .......... .......... .......... .......... ..........  1% 44.1M 3s
   550K .......... .......... .......... .......... ..........  1% 9.82M 3s
   600K .......... .......... .......... .......... ..........  1% 57.9M 3s
   650K ..

✓ Labels ready (117,266 train labels)

[2/3] Downloading COCO val2017 images (~1 GB)...



     0K .......... .......... .......... .......... ..........  0% 1.32M 9m49s
    50K .......... .......... .......... .......... ..........  0% 2.65M 7m21s
   100K .......... .......... .......... .......... ..........  0%  117M 4m56s
   150K .......... .......... .......... .......... ..........  0% 2.68M 4m54s
   200K .......... .......... .......... .......... ..........  0%  108M 3m57s
   250K .......... .......... .......... .......... ..........  0%  176M 3m18s
   300K .......... .......... .......... .......... ..........  0%  173M 2m51s
   350K .......... .......... .......... .......... ..........  0% 2.79M 3m4s
   400K .......... .......... .......... .......... ..........  0% 99.9M 2m44s
   450K .......... .......... .......... .......... ..........  0%  222M 2m28s
   500K .......... .......... .......... .......... ..........  0%  223M 2m15s
   550K .......... .......... .......... .......... ..........  0%  189M 2m4s
   600K .......... .......... .......... .......... .

✓ val2017 downloaded (5,000 images)
      Merging val2017 into val split...


VAL merge:   0%|          | 0/5000 [00:00<?, ?it/s]

✓ val2017 merged (4,952 images)

[3/3] Downloading 60,000 train2017 images ...


TRAIN download:   0%|          | 0/60000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [7]:
# ══════════════════════════════════════════════════════════
# CELL 7: CREATE MERGED DATA.YAML
# ══════════════════════════════════════════════════════════
import yaml
from pathlib import Path

MERGED_DIR = Path('/kaggle/working/merged_dataset')

merged_config = {
    'path'  : str(MERGED_DIR),
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : NC_TOTAL,
    'names' : {i: n for i, n in enumerate(ALL_CLASSES)},
}

YAML_PATH = MERGED_DIR / 'data.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(merged_config, f,
              default_flow_style=False,
              sort_keys=False,
              allow_unicode=True)

print(f"✓ data.yaml written → {YAML_PATH}")
print(f"  nc     = {NC_TOTAL}")
print(f"  COCO classes  : 0–79")
print(f"  New classes   : 80–{NC_TOTAL-1}")
for i, cls in enumerate(TRULY_NEW_CLASSES):
    print(f"    [{80+i}] {cls}")

✓ data.yaml written → /kaggle/working/merged_dataset/data.yaml
  nc     = 89
  COCO classes  : 0–79
  New classes   : 80–88
    [80] Electric pole
    [81] Traffic signs
    [82] Tree
    [83] Uncovered manhole
    [84] crosswalk
    [85] door
    [86] pole
    [87] sidewalk
    [88] stair


In [8]:
# ══════════════════════════════════════════════════════════
# CELL 8: VERIFY MERGED DATASET
# ══════════════════════════════════════════════════════════
import yaml, random, cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

MERGED_DIR = Path('/kaggle/working/merged_dataset')

with open(MERGED_DIR / 'data.yaml') as f:
    cfg = yaml.safe_load(f)

print("=" * 60)
print("MERGED DATASET VERIFICATION")
print("=" * 60)
print(f"nc = {cfg['nc']}")

for split in ['train', 'val']:
    img_dir = MERGED_DIR / 'images' / split
    lbl_dir = MERGED_DIR / 'labels' / split
    imgs = len(list(img_dir.glob('*')))   if img_dir.exists() else 0
    lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    coco = len(list(img_dir.glob('coco_*'))) if img_dir.exists() else 0
    ds1  = len(list(img_dir.glob('ds1_*')))  if img_dir.exists() else 0
    ds2  = len(list(img_dir.glob('ds2_*')))  if img_dir.exists() else 0
    ok   = "✓" if abs(imgs-lbls) <= 10 else "⚠"
    print(f"\n  {ok} {split}: {imgs:,} images | {lbls:,} labels")
    print(f"       ├─ COCO : {coco:,}")
    print(f"       ├─ DS1  : {ds1:,}")
    print(f"       └─ DS2  : {ds2:,}")

# Class distribution (sample 5k labels)
counts = defaultdict(int)
for lbl in list((MERGED_DIR/'labels'/'train').glob('*.txt'))[:5000]:
    for line in open(lbl):
        p = line.strip().split()
        if p:
            counts[int(p[0])] += 1

print("\nNew classes (80+):")
for i, cls in enumerate(TRULY_NEW_CLASSES):
    c   = counts.get(80+i, 0)
    bar = "█" * min(c//10, 30)
    ok  = "✓" if c > 100 else "⚠"
    print(f"  {ok} [{80+i}] {cls:25s}: {c:5d}  {bar}")

print("\nTop 10 COCO classes found:")
top = sorted(((k,v) for k,v in counts.items() if k<80),
             key=lambda x: x[1], reverse=True)[:10]
for cid, cnt in top:
    bar = "█" * min(cnt//50, 30)
    print(f"     [{cid:2d}] {COCO_CLASSES[cid]:25s}: {cnt:5d}  {bar}")

MERGED DATASET VERIFICATION
nc = 89

  ⚠ train: 96,271 images | 94,671 labels
       ├─ COCO : 59,985
       ├─ DS1  : 7,855
       └─ DS2  : 28,431

  ⚠ val: 13,942 images | 13,132 labels
       ├─ COCO : 4,952
       ├─ DS1  : 865
       └─ DS2  : 8,125

New classes (80+):
  ⚠ [80] Electric pole            :    77  ███████
  ⚠ [81] Traffic signs            :    68  ██████
  ✓ [82] Tree                     :   116  ███████████
  ⚠ [83] Uncovered manhole        :    54  █████
  ✓ [84] crosswalk                :   127  ████████████
  ✓ [85] door                     :   163  ████████████████
  ✓ [86] pole                     :   151  ███████████████
  ✓ [87] sidewalk                 :   187  ██████████████████
  ✓ [88] stair                    :   139  █████████████

Top 10 COCO classes found:
     [ 0] person                   :  7454  ██████████████████████████████
     [ 2] car                      :  1565  ██████████████████████████████
     [56] chair                    :  1290  ███

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 9: TRAIN — OPTIMIZED FOR T4 x2, 12-HR SESSION
#
# Key improvements over your previous training cell:
#   ✓ device='0,1'    → uses BOTH T4 GPUs
#   ✓ batch=32        → doubled (scales with 2 GPUs)
#   ✓ lr0=2e-4        → scaled with batch (linear scaling rule)
#   ✓ cos_lr=True     → cosine LR decay, better convergence
#   ✓ close_mosaic=15 → disables mosaic in last 15 epochs
#                        for cleaner fine-tuning
#   ✓ epochs=65      → maximizes the 12-hr session
#   ✓ amp=True        → mixed precision, faster + less VRAM
#   ✓ workers=8       → faster data loading
#   ✓ save_period=1   → checkpoint every epoch to /backups
#   ✓ backup callback → independent copy to Drive/backups
# ══════════════════════════════════════════════════════════
from ultralytics import YOLO
import shutil
from pathlib import Path

BACKUP_DIR = Path('/kaggle/working/backups')
BACKUP_DIR.mkdir(exist_ok=True)

def backup_checkpoint(trainer):
    """Copies last.pt to /backups after every 10 epochs."""
    epoch = trainer.epoch
    src   = Path(trainer.save_dir) / 'weights' / 'last.pt'
    if src.exists():
        dst = BACKUP_DIR / f'last_epoch{epoch+1:03d}.pt'
        shutil.copy2(src, dst)
        # Keep only last 5 backups to save disk space
        old = sorted(BACKUP_DIR.glob('last_epoch*.pt'))[:-5]
        for f in old:
            f.unlink(missing_ok=True)
        print(f"  💾 Epoch {epoch+1:3d} backed up")

model = YOLO('yolov8n.pt')
model.add_callback('on_train_epoch_end', backup_checkpoint)

results = model.train(
    data          = '/kaggle/working/merged_dataset/data.yaml',

    # ── Core settings ────────────────────────────────────
    epochs        = 60,       
    imgsz         = 640,
    batch         = 32,        # 16 per GPU × 2 GPUs
    device        = '0,1',     # BOTH T4s

    # ── Learning rate (linear scaling: lr × batch/16) ────
    lr0           = 1.5e-4,      # 1e-4 × (32/16)
    lrf           = 0.01,      # final lr = lr0 × lrf
    cos_lr        = True,      # cosine decay (smoother than linear)
    warmup_epochs = 3,
    warmup_bias_lr= 0.1,

    # ── Regularization ───────────────────────────────────
    freeze        = 10,        # freeze backbone only (layers 0–9)
    weight_decay  = 0.0005,
    dropout       = 0.0,

    # ── Augmentation ─────────────────────────────────────
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    degrees       = 10.0,
    translate     = 0.1,
    scale         = 0.5,
    flipud        = 0.1,
    fliplr        = 0.5,
    mosaic        = 1.0,
    mixup         = 0.15,
    copy_paste    = 0.1,
    close_mosaic  = 15,        # turn off mosaic last 15 epochs

    # ── Performance ──────────────────────────────────────
    amp           = True,      # mixed precision (faster, less VRAM)
    workers       = 8,         # data loader threads

    # ── Early stopping ───────────────────────────────────
    patience      = 20,        # stop if no improvement for 20 epochs

    # ── Saving ───────────────────────────────────────────
    project       = '/kaggle/working/yolo_blind_assist',
    name          = 'blind_assist_final',
    save          = True,
    save_period   = 10,         # save last.pt every 10 epochs
    plots         = True,
    val           = True,
)

print("\n" + "=" * 55)
print("TRAINING COMPLETE")
print("=" * 55)
print(f"Best weights : {results.save_dir}/weights/best.pt")
print(f"mAP50        : {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.4f}")
print(f"mAP50-95     : {results.results_dict.get('metrics/mAP50-95(B)', 'N/A'):.4f}")

Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/merged_dataset/data.yaml, degrees=10.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=blind_a

In [ ]:
modle.train(resume = true)